# Treino experimental da voz da Lia (Piper Plus / VITS)

**Notebook preparado para execução manual no Google Colab.** Ao executar o passo de upload, os arquivos de voz serão transferidos do seu computador para o runtime Google. Você autorizou esse envio nesta conversa. O notebook não monta Drive, não publica no Hugging Face e não envia áudio para terceiros além do ambiente Colab escolhido. O runtime é temporário; interrompa e apague a sessão ao terminar.

## Condição de licença

Candidato técnico: `ayousanz/piper-plus-base`, um checkpoint multilíngue de seis idiomas com ramo `pt` (ID 5), **não um modelo-base exclusivo de PT-BR**. O G2P `pt` representa o português brasileiro, mas a qualidade depende de os tokens fonéticos caberem no inventário do checkpoint. O model card publica dados/receita, mas não declara claramente a licença dos pesos. O código do Piper Plus se declara MIT, o que **não licencia os pesos**. A autorização dada nesta conversa permite a prova privada/local; o fluxo prossegue sem edição de código. Não publique nem redistribua o checkpoint/base/fine-tune sem confirmar os direitos.

Não use áudio de outra pessoa sem autorização. Revise transcrições manualmente. Este fluxo não faz ASR. Mantenha originais privados fora do Git.

**Entrada:** pode ser o MP3 original, sem você montar ZIP ou CSV. O notebook cria os segmentos WAV e transcreve para um rascunho de `metadata.csv`; você só confere a tabela e confirma. Também aceita um ZIP pronto com `metadata.csv` (`id|texto`) e WAVs mono.

In [ ]:
# @title Aceitar avaliação privada (automático)
# Controles explícitos. Nenhum upload ocorre nesta célula.
PRIVATE_EVALUATION_ONLY = True
ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST = True  # autorização do usuário: avaliar em privado; sem redistribuição

if not PRIVATE_EVALUATION_ONLY or not ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST:
    raise RuntimeError('Este fluxo foi autorizado apenas para avaliação privada, sem redistribuição.')
print('Modo privado autorizado. Nenhum áudio foi enviado ainda; o upload ocorre na célula própria.')

## 1. Ambiente do treino

Instalações ocorrem apenas dentro do Colab, não no Windows nem no repositório. Código fixado no commit consultado em 2026-09-30. **Antes de executar, o notebook solicita GPU T4 automaticamente nos metadados**; o Colab ainda pode oferecer outra GPU ou nenhuma, conforme disponibilidade. Se o runtime iniciar sem GPU, escolha T4 no menu `Runtime > Change runtime type > Hardware accelerator`. Depois use `Runtime > Run all`: ele instala o ambiente, pede o ZIP uma vez, valida os dados, treina, exporta e oferece o resultado para download.

In [ ]:
# @title Preparar Colab com GPU T4 e instalar dependências
import pathlib, subprocess, torch, sys, shutil
# Usa a API do PyTorch; não depende do executável nvidia-smi.
if not torch.cuda.is_available():
    raise RuntimeError('Este runtime Colab não tem GPU disponível. Antes de executar novamente, selecione Runtime > Change runtime type > Hardware accelerator > GPU e reinicie a sessão.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
# Piper-Plus documents espeak-ng as a preprocessing prerequisite.
if not shutil.which('espeak-ng'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','espeak-ng'], check=True)
print('espeak-ng:', shutil.which('espeak-ng'))
REPO = pathlib.Path('/content/piper-plus')
PIN = '03f22b895d6b87d3733223c0844cb2ef6569937b'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/ayutaz/piper-plus.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',PIN], check=True)
subprocess.run(['python','-m','pip','install','uv'], check=True)
subprocess.run(['uv','pip','install','--system','.[train]'], cwd=REPO, check=True)
subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','faster-whisper==1.2.1','av==18.1.0','soundfile'], check=True)
# Confirma no mesmo interpretador antes de liberar a célula de upload.
subprocess.run([sys.executable,'-c','import av, faster_whisper, soundfile; print("PyAV", av.__version__, "Faster-Whisper ready")'], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## 2. Enviar áudio MP3 ou dataset pronto

Escolha seu MP3 diretamente no seletor — não precisa preparar ZIP, WAV ou CSV. O notebook usa Faster-Whisper `small` em CPU, com dependências PyAV instaladas/verificadas antes do upload e pesos MIT pinados, dentro do próprio runtime Colab (sem serviço externo), divide o áudio em segmentos e cria WAVs mono + `metadata.csv`. Se já tiver dataset pronto, também aceita um ZIP com `metadata.csv` e WAVs. O reconhecimento cria rascunhos: confira a tabela antes de confirmar o treino.


In [ ]:
# @title Enviar MP3 e gerar segmentos/transcrição automaticamente
from google.colab import files
from pathlib import Path
import zipfile, shutil, csv, re, subprocess, sys, importlib.util

DATA_ROOT = Path('/content/lia_voice_data')
shutil.rmtree(DATA_ROOT, ignore_errors=True)
DATA_ROOT.mkdir(parents=True)
uploaded = files.upload()  # escolha o MP3; o áudio é enviado somente ao Colab
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
audio_names = [name for name in uploaded if Path(name).suffix.lower() in {'.mp3','.wav','.m4a','.flac','.ogg'}]
if zip_names:
    if len(zip_names) != 1 or audio_names:
        raise ValueError('Envie um ZIP pronto, ou um/mais arquivos de áudio, não os dois.')
    with zipfile.ZipFile(zip_names[0]) as zf:
        root = DATA_ROOT.resolve()
        for info in zf.infolist():
            target = (DATA_ROOT / info.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError('Caminho inseguro dentro do ZIP; extração bloqueada.')
        zf.extractall(DATA_ROOT)
    print('Dataset extraído; seguirá para validação de metadata.csv.')
elif audio_names and len(audio_names) == len(uploaded):
    # Self-heal if this cell is run without first running the setup cell.
    missing = [name for name in ('av','faster_whisper','soundfile') if importlib.util.find_spec(name) is None]
    if missing:
        print('Instalando dependências de áudio ausentes:', ', '.join(missing))
        subprocess.run([sys.executable,'-m','pip','install','faster-whisper==1.2.1','av==18.1.0','soundfile'], check=True)
    import av, numpy as np, soundfile as sf
    from faster_whisper import WhisperModel
    audio_root = DATA_ROOT
    metadata_rows = []
    from huggingface_hub import snapshot_download
    ASR_MODEL_REVISION='536b0662742c02347bc0e980a01041f333bce120'
    ASR_MODEL_PATH=snapshot_download('Systran/faster-whisper-small', revision=ASR_MODEL_REVISION)
    asr = WhisperModel(ASR_MODEL_PATH, device='cpu', compute_type='int8')
    def decode_mono_22050(path):
        chunks=[]
        with av.open(str(path)) as container:
            stream=container.streams.audio[0]
            resampler=av.AudioResampler(format='flt', layout='mono', rate=22050)
            for frame in container.decode(stream):
                for converted in resampler.resample(frame):
                    chunks.append(converted.to_ndarray().reshape(-1).astype(np.float32))
            for converted in resampler.resample(None):
                chunks.append(converted.to_ndarray().reshape(-1).astype(np.float32))
        if not chunks: raise ValueError(f'Não foi possível decodificar {path.name}')
        return np.concatenate(chunks)
    counter=0
    for original_name in audio_names:
        audio_path=Path('/content')/original_name
        print('Transcrevendo em português:', original_name)
        segments, info = asr.transcribe(str(audio_path), language='pt', beam_size=5,
                                        vad_filter=True, word_timestamps=True)
        segments=list(segments)  # consome o gerador e conclui a transcrição
        waveform=decode_mono_22050(audio_path)
        sr=22050
        words=[]
        for segment in segments:
            if segment.words:
                words.extend(segment.words)
            elif segment.text.strip():
                words.append(type('Word',(),{'start':segment.start,'end':segment.end,'word':segment.text})())
        groups=[]; current=[]; group_start=None
        for word in words:
            text=(word.word or '').strip()
            if not text: continue
            if group_start is None: group_start=float(word.start)
            current.append(word)
            elapsed=float(word.end)-group_start
            if text.endswith(('.', '?', '!', '…')) or elapsed >= 8.0:
                groups.append(current); current=[]; group_start=None
        if current: groups.append(current)
        for group in groups:
            start=max(0.0,float(group[0].start)-0.08)
            end=min(len(waveform)/sr,float(group[-1].end)+0.12)
            text=''.join(w.word or '' for w in group).strip().replace('|',',').replace('\n',' ')
            if end-start < 0.8 or not text: continue
            counter+=1
            clip_id=f'fala_{counter:04d}'
            i0=max(0,int(start*sr)); i1=min(len(waveform),int(end*sr))
            sf.write(audio_root/(clip_id+'.wav'),waveform[i0:i1],sr,subtype='PCM_16')
            metadata_rows.append((clip_id,text))
        print(f"Idioma detectado: {info.language}; confiança {info.language_probability:.2f}")
    if not metadata_rows:
        raise ValueError('A transcrição não gerou segmentos utilizáveis.')
    metadata_path=DATA_ROOT/'metadata.csv'
    with metadata_path.open('w',encoding='utf-8',newline='') as f:
        writer=csv.writer(f,delimiter='|',lineterminator='\n',quoting=csv.QUOTE_MINIMAL)
        writer.writerows(metadata_rows)
    print(f'Criados {len(metadata_rows)} segmentos e metadata.csv em {DATA_ROOT}.')
else:
    raise ValueError('Envie um MP3/áudio, ou um ZIP pronto com metadata.csv e WAVs.')


## 3. Validar estrutura e áudio

`metadata.csv` usa pipe (`id|texto`), sem cabeçalho. Cada identificador precisa corresponder a `id.wav`. O pré-processamento Piper converte para 22.050 Hz. Revise transcrições, nomes, números, clipping, ruído e cortes antes do treino. O pré-processador Piper converte para mono/22.050 Hz, normaliza e detecta/recorta silêncio; não faz redução de ruído. A célula seguinte audita os WAVs sem alterar os originais. A célula mostra a tabela dos rascunhos e pausa para você digitar `SIM`; se quiser corrigir algum texto, edite `/content/lia_voice_data/metadata.csv` no painel Files antes de confirmar. Essa é uma revisão de texto, não configuração de código.

In [ ]:
# @title Revisar transcrição no campo de texto e confirmar
import soundfile as sf
import zipfile
DATA_ROOT = Path('/content/lia_voice_data')
meta_files = list(DATA_ROOT.rglob('metadata.csv'))
if len(meta_files) != 1:
    raise ValueError(f'Esperado um metadata.csv; encontrados {len(meta_files)}')
META = meta_files[0]
AUDIO_DIR = META.parent
rows, missing, durations, rates = [], [], [], set()
for line_no, line in enumerate(META.read_text(encoding='utf-8').splitlines(), 1):
    if not line.strip() or line.lstrip().startswith('#'): continue
    parts = line.split('|', 1)
    if len(parts) != 2 or not all(part.strip() for part in parts):
        raise ValueError(f'Linha {line_no} inválida; esperado id|texto')
    key, text = parts[0].strip(), parts[1].strip()
    wav = AUDIO_DIR / (key + '.wav')
    if not wav.is_file():
        missing.append((line_no, key)); continue
    info = sf.info(wav)
    if info.channels != 1 or info.frames == 0:
        raise ValueError(f'{wav.name}: esperado WAV mono não vazio')
    rows.append((key, text)); durations.append(info.duration); rates.add(info.samplerate)
if missing: raise FileNotFoundError(f'Áudios ausentes: {missing[:10]}')
if not rows: raise ValueError('Dataset vazio')
print(f'{len(rows)} frases; {sum(durations)/60:.1f} min; duração {min(durations):.2f}-{max(durations):.2f} s')
print('Sample rates encontrados:', sorted(rates))
print('Validação estrutural concluída. Rascunho de transcrições:')
from IPython.display import display
from google.colab import output, files
import ipywidgets as widgets
output.enable_custom_widget_manager()
print(f'{len(rows)} segmentos. Revise/edite o texto abaixo; somente as transcrições serão alteradas.')
editor=widgets.Textarea(value=META.read_text(encoding='utf-8'), description='metadata.csv',
                        layout=widgets.Layout(width='100%',height='420px'),
                        style={'description_width':'initial'})
display(editor)
approval=input('Após revisar o campo acima, digite SIM para salvar e continuar. Qualquer outra resposta interrompe: ').strip().upper()
if approval != 'SIM':
    raise RuntimeError('Parei antes do treino. Reexecute as células após revisar os dados.')
new_rows=[]
for line_no,line in enumerate(editor.value.splitlines(),1):
    if not line.strip(): continue
    parts=line.split('|',1)
    if len(parts)!=2 or not parts[0].strip() or not parts[1].strip():
        raise ValueError(f'Linha {line_no} inválida; preserve o formato id|texto.')
    new_rows.append((parts[0].strip(),parts[1].strip()))
expected_ids={row[0] for row in rows}
new_ids=[row[0] for row in new_rows]
if len(new_ids)!=len(set(new_ids)) or set(new_ids)!=expected_ids:
    raise ValueError('Não altere os IDs das gravações; corrija somente o texto transcrito.')
with META.open('w',encoding='utf-8',newline='') as f:
    for clip_id,text in new_rows:
        f.write(f'{clip_id}|{text.replace(chr(10)," " )}\n')
rows = list(new_rows)  # Keep later G2P audit aligned with the approved metadata.
print('Transcrições salvas; seguindo para pré-processamento e treino.')
# Archive only the reviewed source WAVs and metadata (not models/cache), so it
# can be downloaded and re-uploaded after Colab resets without repeating ASR.
VOICE_DATA_ARCHIVE = Path('/content/lia_voice_dataset.zip')
with zipfile.ZipFile(VOICE_DATA_ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as zf:
    zf.write(META,arcname=META.relative_to(DATA_ROOT))
    for clip_id,_text in new_rows:
        wav = AUDIO_DIR / (clip_id + '.wav')
        if not wav.is_file():
            raise FileNotFoundError(f'Não foi possível empacotar o áudio revisado: {wav}')
        zf.write(wav,arcname=wav.relative_to(DATA_ROOT))
print(f'Backup ZIP pronto: {VOICE_DATA_ARCHIVE} ({VOICE_DATA_ARCHIVE.stat().st_size/1024/1024:.1f} MiB). Baixando para reutilizar após reset do Colab.')
files.download(str(VOICE_DATA_ARCHIVE))

In [ ]:
# @title Auditar áudio antes do treino (não altera os WAVs)
# Auditoria apenas local e não destrutiva. O Piper já fará normalização e VAD;
# não aplique denoise automaticamente sem ouvir amostras, pois pode metalizar a voz.
from pathlib import Path
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

if 'AUDIO_DIR' not in globals() or 'rows' not in globals():
    raise RuntimeError('Execute primeiro a revisão do dataset; não encontrei AUDIO_DIR/rows.')

AUDIO_QC = []
for clip_id, text in rows:
    wav = Path(AUDIO_DIR) / f'{clip_id}.wav'
    if not wav.is_file():
        raise FileNotFoundError(f'Áudio ausente durante a auditoria: {wav}')
    audio, sr = sf.read(wav, dtype='float32', always_2d=True)
    mono = audio.mean(axis=1)
    peak = float(np.max(np.abs(mono))) if mono.size else 0.0
    rms = float(np.sqrt(np.mean(np.square(mono)))) if mono.size else 0.0
    dbfs = 20 * np.log10(max(rms, 1e-8))
    clipped = float(np.mean(np.abs(mono) >= 0.999)) if mono.size else 0.0
    near_silence = float(np.mean(np.abs(mono) < 0.003)) if mono.size else 1.0
    active = np.flatnonzero(np.abs(mono) >= 0.01)
    lead_silence = (int(active[0]) / sr) if active.size else len(mono) / sr
    tail_silence = ((len(mono)-1-int(active[-1])) / sr) if active.size else 0.0
    duration = len(mono) / sr
    flags = []
    if duration < 0.5: flags.append('curto')
    if clipped > 0.001: flags.append('possível clipping')
    if dbfs < -40: flags.append('nível muito baixo')
    if near_silence > 0.65: flags.append('muito silêncio/baixo nível')
    if lead_silence > 0.5 or tail_silence > 0.7: flags.append('silêncio longo nas bordas')
    AUDIO_QC.append({
        'id':clip_id, 'path':wav, 'text':text, 'sr':sr,
        'channels':audio.shape[1], 'duration':duration, 'peak':peak,
        'rms_dbfs':float(dbfs), 'clipped_fraction':clipped,
        'near_silence_fraction':near_silence,
        'lead_silence':lead_silence, 'tail_silence':tail_silence,
        'flags':flags,
    })

flagged = [item for item in AUDIO_QC if item['flags']]
print(f'Auditados {len(AUDIO_QC)} WAVs; sinalizados: {len(flagged)}.')
print('Este relatório não modifica nem normaliza os arquivos-fonte.')
print('Critérios são triagem, não prova automática de áudio ruim.')
for item in flagged[:20]:
    print(
        f"{item['id']}: {', '.join(item['flags'])}; "
        f"{item['duration']:.2f}s, {item['sr']}Hz, {item['channels']} canal(is), "
        f"RMS {item['rms_dbfs']:.1f} dBFS, clipping {item['clipped_fraction']:.3%}; "
        f"texto={item['text']}"
    )

if flagged:
    print('Ouça até três amostras sinalizadas antes de aprovar o pré-processamento:')
    for item in flagged[:3]:
        print(item['id'], '|', item['text'])
        display(Audio(filename=str(item['path'])))
    decision = input(
        'Após ouvir, digite CONTINUAR para usar os WAVs originais com o VAD/normalizador do Piper. '
        'Qualquer outra resposta interrompe para revisar o áudio/transcrição: '
    ).strip().upper()
    if decision != 'CONTINUAR':
        raise RuntimeError('Pré-processamento pausado para revisão dos WAVs sinalizados.')
else:
    print('Nenhum WAV foi sinalizado pela triagem automática.')

# Enforce basic technical validity without rewriting source audio.
invalid = [item for item in AUDIO_QC if item['sr'] <= 0 or item['duration'] <= 0 or item['channels'] < 1]
if invalid:
    raise RuntimeError(f'WAVs tecnicamente inválidos: {[item["id"] for item in invalid]}')
print('Auditoria concluída; os WAVs originais permanecem intactos.')


## 4. Localizar base, revisar licença e preparar

O notebook baixa automaticamente o checkpoint e `config.json` da revisão pinada do model card. Verifica SHA-256 e tamanho do checkpoint. O arquivo usa formato PyTorch checkpoint/pickle: só confie no artefato e no código upstream fixados. A licença do peso não está clara; conforme sua autorização, o uso fica restrito a avaliação privada, sem publicação ou redistribuição.

In [ ]:
# @title Baixar e verificar a base (automático)
from huggingface_hub import hf_hub_download
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
BASE_REPO = 'ayousanz/piper-plus-base'
BASE_REVISION = 'ce006e5da7851fdc469887026eeb3f2175e7e080'
BASE_CHECKPOINT = hf_hub_download(BASE_REPO, 'model.ckpt', revision=BASE_REVISION)
BASE_CONFIG = hf_hub_download(BASE_REPO, 'config.json', revision=BASE_REVISION)
EXPECTED_SHA256 = '83c079abc50575eeb837e1d21e98aefafb23981b9f2cbee247cbcd222df6216b'
sha = hashlib.sha256(Path(BASE_CHECKPOINT).read_bytes()).hexdigest()
if sha != EXPECTED_SHA256:
    raise ValueError(f'Hash inesperado do checkpoint: {sha}')
DATASET_DIR = '/content/lia_dataset'
OUT_DIR = '/content/lia_voice_experiment'
print('Checkpoint verificado:', sha, '| bytes:', Path(BASE_CHECKPOINT).stat().st_size)
base_config = json.loads(Path(BASE_CONFIG).read_text(encoding='utf-8'))
PREPROCESS_LANGUAGE = base_config.get('language', {}).get('code', '')
if not PREPROCESS_LANGUAGE or 'pt' not in PREPROCESS_LANGUAGE.split('-'):
    raise ValueError(f'O checkpoint não declara português no inventário de línguas: {PREPROCESS_LANGUAGE!r}')
CHECKPOINT_PHONEME_MAP = base_config.get('phoneme_id_map', {})
CHECKPOINT_NUM_SYMBOLS = base_config.get('num_symbols')
if not CHECKPOINT_PHONEME_MAP or not isinstance(CHECKPOINT_NUM_SYMBOLS,int):
    raise ValueError('O config do checkpoint não contém phoneme_id_map/num_symbols utilizáveis.')
checkpoint_ids = [int(i) for values in CHECKPOINT_PHONEME_MAP.values() for i in values]
if len(CHECKPOINT_PHONEME_MAP) != CHECKPOINT_NUM_SYMBOLS or sorted(checkpoint_ids) != list(range(CHECKPOINT_NUM_SYMBOLS)):
    raise ValueError('O mapa fonêmico do checkpoint não é uma bijeção densa; abortando para evitar IDs incompatíveis.')
CHECKPOINT_PHONEME_MAP_PATH = Path('/content/lia_checkpoint_phoneme_id_map.json')
CHECKPOINT_PHONEME_MAP_PATH.write_text(json.dumps(CHECKPOINT_PHONEME_MAP,ensure_ascii=False),encoding='utf-8')
os.environ['PIPER_CHECKPOINT_PHONEME_ID_MAP'] = str(CHECKPOINT_PHONEME_MAP_PATH)
print(f"Inventário fonêmico pinado ao checkpoint: {CHECKPOINT_NUM_SYMBOLS} símbolos; ordem de idiomas: {PREPROCESS_LANGUAGE}")
# Piper-Plus otherwise routes Latin script to English whenever 'en' is in the
# six-language model. Apply this patch here, every time, so it cannot be skipped
# when resuming a Colab session or running only the preprocessing cell.
PREPROCESS_SOURCE = Path(REPO) / 'src/python/piper_train/preprocess.py'
# `python -m piper_train.preprocess` resolves the installed module, normally
# in site-packages, rather than the checkout source file. Patch both copies.
INSTALLED_PREPROCESS_SOURCE = Path(subprocess.check_output(
    [sys.executable,'-c','import piper_train.preprocess as p; print(p.__file__)'],
    text=True
).strip()).resolve()
PT_PATCH_MARKER = '# Arena notebook: default Latin G2P to Portuguese for PT fine-tuning.'
MAP_PATCH_MARKER = '# Arena notebook: use exact checkpoint phoneme IDs for preprocessing.'
def patch_preprocess_language(path):
    source = path.read_text(encoding='utf-8')
    changed = False
    if PT_PATCH_MARKER not in source:
        old_phonemizer = '    phonemizer = MultilingualPhonemizer(lang_parts)\n'
        new_phonemizer = (PT_PATCH_MARKER + '\n'
            '    default_latin_language = ("pt" if "pt" in lang_parts else "en" if "en" in lang_parts else lang_parts[0])\n'
            '    phonemizer = MultilingualPhonemizer(lang_parts, default_latin_language=default_latin_language)\n')
        old_detector = ('                default_latin_language="en"\n'
            '                if "en" in args.lang_parts\n'
            '                else args.lang_parts[0],\n')
        new_detector = ('                default_latin_language="pt"\n'
            '                if "pt" in args.lang_parts\n'
            '                else ("en" if "en" in args.lang_parts else args.lang_parts[0]),\n')
        if source.count(old_phonemizer) != 1 or source.count(old_detector) != 1:
            raise RuntimeError(f'{path} divergiu do preprocess.py pinado; patch PT cancelado.')
        source = source.replace(old_phonemizer,new_phonemizer).replace(old_detector,new_detector)
        changed = True
    if MAP_PATCH_MARKER not in source:
        old_map = ('        multilingual_id_map = get_phoneme_id_map("-".join(sorted(lang_parts)))\n'
            '        args.phoneme_id_map = multilingual_id_map\n')
        new_map = (MAP_PATCH_MARKER + '\n'
            '        multilingual_id_map = get_phoneme_id_map("-".join(sorted(lang_parts)))\n'
            '        checkpoint_map_path = os.environ.get("PIPER_CHECKPOINT_PHONEME_ID_MAP")\n'
            '        if checkpoint_map_path:\n'
            '            with open(checkpoint_map_path, encoding="utf-8") as map_file:\n'
            '                multilingual_id_map = json.load(map_file)\n'
            '        args.phoneme_id_map = multilingual_id_map\n')
        if source.count(old_map) != 1:
            raise RuntimeError(f'{path} divergiu do preprocess.py pinado; patch do inventário fonêmico cancelado.')
        source = source.replace(old_map,new_map)
        changed = True
    if changed:
        path.write_text(source,encoding='utf-8')
for path in {PREPROCESS_SOURCE.resolve(), INSTALLED_PREPROCESS_SOURCE}:
    patch_preprocess_language(path)
    patched = path.read_text(encoding='utf-8')
    if PT_PATCH_MARKER not in patched or MAP_PATCH_MARKER not in patched:
        raise RuntimeError(f'Patch de idioma/mapa não confirmado em {path}.')
print('Módulos PT preparados: checkout e módulo instalado usado pelo subprocesso:', INSTALLED_PREPROCESS_SOURCE)

# The pinned G2P emits Portuguese nasal /ɐ̃/ as the single-character token "ã".
# This checkpoint's fixed inventory stores the same phone as PUA U+E064 (ID 153).
# Normalize before encoding in BOTH training preprocessing and ONNX inference.
PT_NASAL_A_MARKER = '# Arena PT-BR: normalize nasal ã to checkpoint token ɐ̃ before encoding.'
def patch_nasal_a(path, old_line, replacement_line):
    source = path.read_text(encoding='utf-8')
    if PT_NASAL_A_MARKER in source:
        return
    if source.count(old_line) != 1:
        raise RuntimeError(f'Layout inesperado em {path}; patch de ã cancelado.')
    path.write_text(source.replace(old_line, replacement_line, 1), encoding='utf-8')

PREPROCESS_NASAL_OLD = '                    raw_tokens, raw_prosody = phonemizer.phonemize_with_prosody(text)\n'
PREPROCESS_NASAL_NEW = (PREPROCESS_NASAL_OLD
    + '                    ' + PT_NASAL_A_MARKER + '\n'
    + '                    raw_tokens = ["ɐ̃" if token == "ã" else token for token in raw_tokens]\n')
INFER_NASAL_OLD = '    phonemes, prosody_info_list = phonemizer.phonemize_with_prosody(text)\n'
INFER_NASAL_NEW = (INFER_NASAL_OLD
    + '    ' + PT_NASAL_A_MARKER + '\n'
    + '    phonemes = ["ɐ̃" if token == "ã" else token for token in phonemes]\n')
INFER_SOURCE = Path(subprocess.check_output(
    [sys.executable,'-c','import piper_train.infer_onnx as m; print(m.__file__)'],text=True
).strip()).resolve()
for path in {PREPROCESS_SOURCE.resolve(), INSTALLED_PREPROCESS_SOURCE}:
    patch_nasal_a(path, PREPROCESS_NASAL_OLD, PREPROCESS_NASAL_NEW)
for path in {Path(REPO)/'src/python/piper_train/infer_onnx.py', INFER_SOURCE}:
    patch_nasal_a(path.resolve(), INFER_NASAL_OLD, INFER_NASAL_NEW)

# Spaces and punctuation are absent from this fixed inventory. The pinned
# encoder used to silently omit those tokens but truncate prosody from the end,
# desynchronizing phone/prosody arrays. Filter known boundary tokens together
# with their prosody records; fail closed for any other unsupported token.
PT_BOUNDARY_MARKER = '# Arena PT-BR: filter unencoded boundaries with aligned prosody.'
def patch_boundaries(path, old_block, new_block):
    source = path.read_text(encoding='utf-8')
    if PT_BOUNDARY_MARKER in source:
        return
    if source.count(old_block) != 1:
        raise RuntimeError(f'Layout inesperado em {path}; patch de limites cancelado.')
    path.write_text(source.replace(old_block, new_block, 1), encoding='utf-8')

PREPROCESS_BOUNDARY_OLD = (
    '                    # PiperEncoder: PUA map -> phoneme_ids -> BOS/EOS/padding\n'
    '                    utt.phoneme_ids, prosody_out = encoder.encode_with_prosody(\n'
)
PREPROCESS_BOUNDARY_NEW = (
    '                    ' + PT_BOUNDARY_MARKER + '\n'
    '                    _kept_tokens, _kept_prosody = [], []\n'
    '                    _unencoded_boundaries = {" ", ".", ",", "!", ";", ":", "…"}\n'
    '                    for _token, _prosody in zip(clean_tokens, clean_prosody, strict=True):\n'
    '                        _mapped_token = map_token(_token)\n'
    '                        if _mapped_token in args.phoneme_id_map:\n'
    '                            _kept_tokens.append(_token)\n'
    '                            _kept_prosody.append(_prosody)\n'
    '                        elif _token in _unencoded_boundaries:\n'
    '                            continue\n'
    '                        else:\n'
    '                            raise ValueError(f"Unsupported content phoneme {_token!r}; refusing silent drop")\n'
    '                    clean_tokens, clean_prosody = _kept_tokens, _kept_prosody\n'
    '                    # PiperEncoder: PUA map -> phoneme_ids -> BOS/EOS/padding\n'
    '                    utt.phoneme_ids, prosody_out = encoder.encode_with_prosody(\n'
)
INFER_BOUNDARY_OLD = (
    '    # Arena PT-BR: normalize nasal ã to checkpoint token ɐ̃ before encoding.\n'
    '    phonemes = ["ɐ̃" if token == "ã" else token for token in phonemes]\n'
    '\n'
    '    from piper_plus_g2p.encode.encoder import PiperEncoder  # noqa: PLC0415\n'
)
INFER_BOUNDARY_NEW = (
    '    # Arena PT-BR: normalize nasal ã to checkpoint token ɐ̃ before encoding.\n'
    '    phonemes = ["ɐ̃" if token == "ã" else token for token in phonemes]\n'
    '    ' + PT_BOUNDARY_MARKER + '\n'
    '    from piper_plus_g2p.encode.pua import map_token  # noqa: PLC0415\n'
    '    _unencoded_boundaries = {" ", ".", ",", "!", ";", ":", "…"}\n'
    '    _kept_phonemes, _kept_prosody = [], []\n'
    '    for _token, _prosody in zip(phonemes, prosody_info_list, strict=True):\n'
    '        _mapped_token = map_token(_token)\n'
    '        if _mapped_token in phoneme_id_map:\n'
    '            _kept_phonemes.append(_token)\n'
    '            _kept_prosody.append(_prosody)\n'
    '        elif _token in _unencoded_boundaries:\n'
    '            continue\n'
    '        else:\n'
    '            raise ValueError(f"Unsupported content phoneme {_token!r}; refusing silent drop")\n'
    '    phonemes, prosody_info_list = _kept_phonemes, _kept_prosody\n'
    '\n'
    '    from piper_plus_g2p.encode.encoder import PiperEncoder  # noqa: PLC0415\n'
)
for path in {PREPROCESS_SOURCE.resolve(), INSTALLED_PREPROCESS_SOURCE}:
    patch_boundaries(path, PREPROCESS_BOUNDARY_OLD, PREPROCESS_BOUNDARY_NEW)
for path in {Path(REPO)/'src/python/piper_train/infer_onnx.py', INFER_SOURCE}:
    patch_boundaries(path.resolve(), INFER_BOUNDARY_OLD, INFER_BOUNDARY_NEW)
print('Tokens de ã e alinhamento dos limites preparados em treino e inferência.')

G2P_SOURCE = Path(REPO) / 'src/python/g2p'
if str(G2P_SOURCE) not in sys.path:
    sys.path.insert(0,str(G2P_SOURCE))
from piper_plus_g2p.multilingual import MultilingualPhonemizer
pt_phonemizer = MultilingualPhonemizer(
    ['ja','en','zh','es','fr','pt'], default_latin_language='pt'
)
pt_tokens,_ = pt_phonemizer.phonemize_with_prosody('Olá, este é um teste em português.')
if not pt_tokens:
    raise RuntimeError('O fonemizador português não produziu fonemas; o pré-processamento não será iniciado.')
print('G2P PT verificado:', ' '.join(pt_tokens[:12]), '| idioma padrão: pt | mapa de 6 idiomas preservado')
from collections import Counter
from piper_plus_g2p.encode.pua import map_token
nasal_a_symbol = map_token('ɐ̃')
if nasal_a_symbol not in CHECKPOINT_PHONEME_MAP:
    raise RuntimeError('O alias PT-BR ɐ̃ não está no mapa fixo do checkpoint; pré-processamento bloqueado.')
BOUNDARY_TOKENS = {' ', '.', ',', '!', '?', ';', ':', '…', '$', '?!', '?.', '?~'}
phoneme_unmapped = Counter()
boundary_unmapped = Counter()
nasal_a_occurrences = 0
for _clip_id, _text in rows:
    _tokens, _prosody = pt_phonemizer.phonemize_with_prosody(_text)
    for _token in _tokens:
        if _token == 'ã':
            nasal_a_occurrences += 1
            _token = 'ɐ̃'
        _mapped = map_token(_token)
        if _mapped not in CHECKPOINT_PHONEME_MAP:
            if _token in BOUNDARY_TOKENS:
                boundary_unmapped[_token] += 1
            else:
                phoneme_unmapped[_token] += 1
if phoneme_unmapped:
    raise RuntimeError(
        'Fonemas de conteúdo não representáveis no checkpoint; treino bloqueado: '
        + repr(dict(phoneme_unmapped))
    )
print(f'G2P PT auditado em {len(rows)} transcrições: {nasal_a_occurrences} tokens ã serão codificados como ɐ̃ (ID {CHECKPOINT_PHONEME_MAP[nasal_a_symbol]}).')
if boundary_unmapped:
    print('Delimitadores sem ID: serão filtrados junto com suas posições de prosódia, sem remapeamento arbitrário:',dict(boundary_unmapped))
if shutil.which('espeak-ng') is None:
    print('Instalando espeak-ng, dependência documentada para preprocessing...')
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','espeak-ng'], check=True)
if len(rows) < 2:
    raise ValueError('O pré-processamento do Piper-Plus requer pelo menos 2 frases; verifique metadata.csv e os WAVs.')
# Piper-Plus computes batch_size as utterances/(workers*2); cap workers so a
# small Colab dataset cannot produce batch_size=0, and avoid over-forking CPUs.
PREPROCESS_WORKERS = max(1, min(4, len(rows) // 2))
# Remove partial output from a previous failed attempt; preprocess appends JSONL.
shutil.rmtree(DATASET_DIR, ignore_errors=True)
preprocess_cmd = [
    sys.executable,'-u','-m','piper_train.preprocess',
    '--input-dir',str(AUDIO_DIR),'--output-dir',DATASET_DIR,
    '--language',PREPROCESS_LANGUAGE,'--dataset-format','ljspeech','--sample-rate','22050',
    '--single-speaker','--phoneme-type','multilingual','--max-workers',str(PREPROCESS_WORKERS)
]
print('Línguas do checkpoint:', PREPROCESS_LANGUAGE)
print('Frases:',len(rows),'| workers:',PREPROCESS_WORKERS)
print('Pré-processamento proposto:', ' '.join(preprocess_cmd))
PREPROCESS_LOG = Path('/content/lia_preprocess.log')

def run_preprocess(command, label):
    result = subprocess.run(command, cwd=REPO, env=os.environ.copy(), text=True, capture_output=True)
    with PREPROCESS_LOG.open('a',encoding='utf-8') as log:
        log.write(f'\n===== {label} | exit={result.returncode} =====\n')
        log.write(result.stdout or '')
        log.write(result.stderr or '')
    if result.stdout:
        print(result.stdout, end='' if result.stdout.endswith('\n') else '\n')
    if result.stderr:
        print(result.stderr, end='' if result.stderr.endswith('\n') else '\n')
    if result.returncode:
        raise RuntimeError(f'Pré-processamento terminou com código {result.returncode}; consulte também {PREPROCESS_LOG}.')
    manifest = Path(DATASET_DIR) / 'dataset.jsonl'
    return sum(1 for line in manifest.open(encoding='utf-8') if line.strip()) if manifest.is_file() else 0

RUN_PREPROCESS = True
if RUN_PREPROCESS:
    PREPROCESS_LOG.write_text('Diagnóstico local do pré-processamento Piper-Plus\n',encoding='utf-8')
    processed = run_preprocess(preprocess_cmd,'tentativa normal')
    if processed == 0:
        # The pinned Piper-Plus worker suppresses worker stderr unless --debug is set.
        # Retry with one worker so the original per-utterance exception is visible.
        print('O Piper-Plus terminou sem exemplos. Repetindo automaticamente com --debug e 1 worker para revelar o erro interno...')
        shutil.rmtree(DATASET_DIR,ignore_errors=True)
        debug_cmd = list(preprocess_cmd)
        debug_cmd[debug_cmd.index('--max-workers')+1] = '1'
        debug_cmd.append('--debug')
        processed = run_preprocess(debug_cmd,'tentativa diagnóstica --debug')
    if processed == 0:
        raise RuntimeError(f'O pré-processamento não gerou nenhum exemplo. Veja o erro detalhado acima e o log {PREPROCESS_LOG}; o treino foi interrompido antes de iniciar.')
    generated_config = json.loads((Path(DATASET_DIR)/'config.json').read_text(encoding='utf-8'))
    if generated_config.get('phoneme_id_map') != CHECKPOINT_PHONEME_MAP or generated_config.get('num_symbols') != CHECKPOINT_NUM_SYMBOLS:
        raise RuntimeError('O config do dataset não corresponde ao inventário exato do checkpoint; treino bloqueado.')
    if generated_config.get('language_id_map') != base_config.get('language_id_map'):
        raise RuntimeError('A ordem/IDs de idiomas do dataset divergem do checkpoint; treino bloqueado.')
    print(f'Pré-processamento validado: {processed} exemplo(s); mapa fonêmico e IDs de idioma conferem com o checkpoint.')
    if processed < len(rows):
        print(f'Aviso: {len(rows)-processed} de {len(rows)} frases foram ignoradas; consulte os avisos do Piper-Plus acima.')
else:
    print('Pré-processamento não iniciado.')

## 5. Treino

A base é multilíngue, não uma voz-base exclusiva de PT-BR. Antes do treino, a célula de preparação valida todas as transcrições contra o mapa fonêmico, normaliza o nasal brasileiro `ã` para `ɐ̃` (ID 153) tanto no treino quanto na inferência, e bloqueia o treino se algum fonema de conteúdo continuar sem ID. Espaços e pontuação sem ID são filtrados junto com suas posições de prosódia; fonemas desconhecidos bloqueiam o treino. A receita segue os parâmetros publicados para Piper Plus multilingual, com 500 épocas e checkpoints de retomada a cada 25 épocas. O treino inicia automaticamente após upload e validações. A duração depende do tamanho do corpus e da GPU alocada; o Colab pode interromper sessões gratuitas.

In [ ]:
# @title Treinar a voz (automático)
import json, os, subprocess, sys
from pathlib import Path

train_config_path = Path(DATASET_DIR) / 'config.json'
train_manifest_path = Path(DATASET_DIR) / 'dataset.jsonl'
if not train_config_path.is_file() or not train_manifest_path.is_file():
    raise FileNotFoundError(f'Dataset pré-processado incompleto: esperado {train_config_path} e {train_manifest_path}')
train_config = json.loads(train_config_path.read_text(encoding='utf-8'))
# Fail closed: never start training when the input IDs cannot index the checkpoint embeddings.
expected_map = base_config.get('phoneme_id_map', {})
if train_config.get('phoneme_id_map') != expected_map or train_config.get('num_symbols') != base_config.get('num_symbols'):
    raise RuntimeError(f"Treino bloqueado: mapa de símbolos dataset={train_config.get('num_symbols')} vs checkpoint={base_config.get('num_symbols')}.")
if train_config.get('language_id_map') != base_config.get('language_id_map'):
    raise RuntimeError('Treino bloqueado: language_id_map/ordem dos seis idiomas diverge do checkpoint.')
train_examples = sum(1 for line in train_manifest_path.open(encoding='utf-8') if line.strip())
if train_examples < 2:
    raise ValueError(f'Dataset de treino insuficiente: {train_examples} exemplos em {train_manifest_path}. Reexecute a célula de pré-processamento; ela agora repete automaticamente com --debug e salva o diagnóstico em /content/lia_preprocess.log.')
# Final safety check against the actual phonemes written by preprocessing.
from collections import Counter
from piper_plus_g2p.encode.pua import map_token
boundary_tokens = {' ', '.', ',', '!', '?', ';', ':', '…', '$', '?!', '?.', '?~'}
unmapped_manifest = Counter()
boundary_manifest = Counter()
for line in train_manifest_path.open(encoding='utf-8'):
    if not line.strip():
        continue
    example = json.loads(line)
    for token in example.get('phonemes') or []:
        mapped = map_token(token)
        if mapped not in expected_map:
            if token in boundary_tokens:
                boundary_manifest[token] += 1
            else:
                unmapped_manifest[token] += 1
if unmapped_manifest:
    raise RuntimeError(
        'Treino bloqueado: o manifesto ainda contém fonemas de conteúdo sem ID: '
        + repr(dict(unmapped_manifest))
    )
print('Auditoria do manifesto: nenhum fonema de conteúdo será descartado.')
if boundary_manifest:
    print('Delimitadores sem ID no inventário-base (não são fonemas; mantidos sem remapeamento):',dict(boundary_manifest))
print(f"Dataset: {train_examples} exemplos | speakers={train_config.get('num_speakers')} | languages={train_config.get('num_languages')} | symbols={train_config.get('num_symbols')}")
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__,'| CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'indisponível')

train_cmd = [
    sys.executable,'-u','-m','piper_train','--dataset-dir',DATASET_DIR,
    '--prosody-dim','16','--accelerator','gpu','--devices','1',
    '--precision','32-true','--max_epochs','500','--batch-size','4',
    '--samples-per-speaker','4','--checkpoint-epochs','25',
    '--base_lr','2e-5','--disable_auto_lr_scaling','--ema-decay','0.9995',
    '--max-phoneme-ids','400','--no-wavlm','--no-compile','--num-workers','2',
    '--resume-from-multispeaker-checkpoint',BASE_CHECKPOINT,
    '--default_root_dir',OUT_DIR
]
print('Comando piloto:', ' '.join(train_cmd))
RUN_TRAINING = True
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Este Colab está sem GPU NVIDIA. Selecione Runtime > Change runtime type > GPU, reconecte e execute as células novamente.')
    # Sem login nem envio de métricas a serviço externo (modo de avaliação privada).
    # TensorBoard continua disponível localmente em OUT_DIR/lightning_logs.
    train_env = os.environ.copy()
    train_env.update({'WANDB_MODE':'disabled','PYTHONUNBUFFERED':'1'})
    train_log = Path(OUT_DIR) / 'training.log'
    Path(OUT_DIR).mkdir(parents=True,exist_ok=True)
    print('W&B desativado; saída completa também será salva em:',train_log)
    with train_log.open('w',encoding='utf-8') as log_file:
        process = subprocess.Popen(
            train_cmd,cwd=REPO,env=train_env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end='',flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code:
        tail=''.join(train_log.read_text(encoding='utf-8',errors='replace').splitlines(keepends=True)[-80:])
        raise RuntimeError(f'Treino terminou com código {return_code}. Últimas linhas do log:\n{tail}\nLog completo: {train_log}')
else:
    print('Treino não iniciado.')


## 6. Exportar, empacotar e baixar

Depois que o treino terminar, a próxima célula encontra o checkpoint mais recente, exporta ONNX, inclui o JSON de configuração e abre o download do pacote. Nenhum arquivo é enviado/publicado automaticamente. Guarde o ZIP fora do repositório; a licença dos pesos segue sem esclarecimento, então o uso continua privado.

Ao terminar, interrompa e exclua o runtime Colab. O notebook não monta Drive.


In [ ]:
# @title Exportar e baixar a voz treinada (automático)
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
from google.colab import files

ckpts = [p for p in Path(OUT_DIR).rglob('*.ckpt') if p.is_file()]
if not ckpts:
    raise FileNotFoundError('Treino finalizado, mas não encontrei checkpoint .ckpt em '+OUT_DIR)
latest = max(ckpts, key=lambda p: p.stat().st_mtime)
EXPORT_DIR = Path(OUT_DIR) / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
ONNX = EXPORT_DIR / 'lia_voice.onnx'
if not latest.is_file() or latest.stat().st_size == 0:
    raise FileNotFoundError(f'Checkpoint vazio ou inexistente: {latest}')
# PT is language ID 5. Preserve its fine-tuned embedding; unifying from default lang[0] (JA) overwrites it.
pt_language_id = int(base_config['language_id_map']['pt'])
if pt_language_id != 5:
    raise RuntimeError(f'ID PT inesperado no checkpoint: {pt_language_id}; exportação abortada.')
export_cmd = [sys.executable,'-u','-m','piper_train.export_onnx','--no-unify-emb-lang',str(latest),str(ONNX)]
print('Checkpoint selecionado:', latest, '| bytes:', latest.stat().st_size)
print('Comando de exportação:', ' '.join(export_cmd))
# The pinned exporter may load Lightning checkpoint weights onto CUDA, but its
# dummy ONNX inputs are CPU tensors. Force the model to CPU before tracing so
# emb_lang and dummy lid share a device. This changes export placement only.
EXPORTER_SOURCE = Path(subprocess.check_output(
    [sys.executable,'-c','import piper_train.export_onnx as e; print(e.__file__)'],text=True
).strip()).resolve()
EXPORT_CPU_MARKER = '# Arena notebook: keep ONNX export model and dummy inputs on CPU.'
exporter_source = EXPORTER_SOURCE.read_text(encoding='utf-8')
if EXPORT_CPU_MARKER not in exporter_source:
    old_load = ('    model = VitsModel.load_from_checkpoint(args.checkpoint, dataset=None, strict=False)\n'
        '    model_g = model.model_g\n')
    new_load = (EXPORT_CPU_MARKER + '\n'
        '    model = VitsModel.load_from_checkpoint(args.checkpoint, dataset=None, strict=False)\n'
        '    model.cpu()\n'
        '    model_g = model.model_g\n')
    if exporter_source.count(old_load) != 1:
        raise RuntimeError(f'Layout inesperado do exportador instalado: {EXPORTER_SOURCE}; exportação abortada.')
    EXPORTER_SOURCE.write_text(exporter_source.replace(old_load,new_load),encoding='utf-8')
if EXPORT_CPU_MARKER not in EXPORTER_SOURCE.read_text(encoding='utf-8'):
    raise RuntimeError('Patch CPU do exportador não foi confirmado; exportação abortada.')
print('Exportador preparado para CPU:', EXPORTER_SOURCE)
EXPORT_LOG = EXPORT_DIR / 'export.log'
export_env = os.environ.copy()
export_env.update({'WANDB_MODE':'disabled','PYTHONUNBUFFERED':'1'})
with EXPORT_LOG.open('w',encoding='utf-8') as log_file:
    process = subprocess.Popen(
        export_cmd,cwd=REPO,env=export_env,stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,text=True,bufsize=1
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line,end='',flush=True)
        log_file.write(line)
        log_file.flush()
    export_return_code = process.wait()
if export_return_code:
    tail=''.join(EXPORT_LOG.read_text(encoding='utf-8',errors='replace').splitlines(keepends=True)[-100:])
    raise RuntimeError(f'Exportação ONNX terminou com código {export_return_code}. Log completo: {EXPORT_LOG}\nÚltimas linhas:\n{tail}')
if not ONNX.is_file() or ONNX.stat().st_size == 0:
    raise FileNotFoundError(f'Exportador terminou sem criar um ONNX válido: {ONNX}; log: {EXPORT_LOG}')

# O exportador normalmente cria o JSON pareado. Se não o fizer, usar o config
# da mesma base pinada, cujo inventário de fonemas/arquitetura foi validado.
config_candidates = [Path(str(ONNX)+'.json'), ONNX.with_suffix('.onnx.json'), EXPORT_DIR/'config.json']
config_out = next((x for x in config_candidates if x.is_file()), None)
if config_out is None:
    config_out = Path(str(ONNX)+'.json')
    shutil.copy2(BASE_CONFIG, config_out)

package = Path('/content/lia_voice_private_eval.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX, ONNX.name)
    zf.write(config_out, 'lia_voice.onnx.json')
with zipfile.ZipFile(package) as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f'ZIP de exportação corrompido no item {bad_member}')
    if ONNX.name not in zf.namelist() or 'lia_voice.onnx.json' not in zf.namelist():
        raise RuntimeError('ZIP incompleto: esperado ONNX e lia_voice.onnx.json.')
print('Pacote pronto e verificado:', package, '| bytes:', package.stat().st_size)
files.download(str(package))
